1. Read the file using spark dataframe reader API
2. Add metadata columns - Source file, Ingestion timestamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
circuit_schema = "circuitId string, url string, circuitName string, lat double, long double, locality string, country string"

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option("header", "true")
        .schema(circuit_schema)
        .load(source_file)
)

display(circuits_df)

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

display(circuits_final_df)

### Write to bronze delta table

In [0]:
write_to_bronze(
    input_df = circuits_final_df
    , target_table = table_name
    , batch_id = v_batch_id
)

In [0]:
%sql
select * from formula1_incr.bronze.circuits

In [0]:
display(spark.table(table_name))